# G3.1 — M02 from library components: check against Python

Exercise card: `G3.1_zone_rc_composition.md` · 20 min. The model is written in **OMEdit**; this
notebook simulates your Modelica file and compares it with the Python M02 of Session 2.

**Pass criterion.** Largest deviation from the Python M02 below 0.01 K for both temperatures.

The Modelica file must contain a model `S03_G1_ZoneRC` with components `zone` and `envelope`
(MSL `HeatCapacitor`), as in the starting file. Cells tagged `omc` run OpenModelica from Python with `energy_training.omc`: a local `omc` if OpenModelica is installed, otherwise the course Docker image. Each simulation takes a few seconds (the model is translated to C and compiled first).

Run the cells from the top (*Shift+Enter*); write your answers in the cells marked *Your answer*.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from energy_training.omc.results import read_result
from energy_training.omc.runner import flatten, simulate
from energy_training.plotting.style import apply_style, degc, hours
from energy_training.simulation import zone_rc as zr
from energy_training.utilities import repo_root

apply_style()
OUT = repo_root() / "output/s03_g1"  # git-ignored
T_AMB, Q_BEFORE, Q_AFTER = 273.15, 1000.0, 4000.0  # as in the starting file
TOLERANCE_K = 0.01


def compare(result):
    """Largest absolute deviation [K] of zone and envelope temperature from the Python M02."""
    p = zr.ZoneRCParams()
    t = result.index.to_numpy(dtype=float)
    ref = zr.simulate(
        (0.0, float(t[-1])), zr.steady_state(Q_BEFORE, T_AMB, p), Q_AFTER, T_AMB, p, t_eval=t
    )
    return {
        "zone": float(np.abs(result["zone.T"].to_numpy() - ref.T_z).max()),
        "envelope": float(np.abs(result["envelope.T"].to_numpy() - ref.T_e).max()),
    }, ref


def check(mo_file):
    """Simulate model S03_G1_ZoneRC in mo_file, print the deviations, plot both models."""
    run = simulate("S03_G1_ZoneRC", [Path(mo_file).resolve()], out_dir=OUT, load_library=False)
    result = read_result(run.csv)
    dev, ref = compare(result)
    for name, d in dev.items():
        print(f"max |T_{name} (Modelica) - T_{name} (Python)| = {d:.2e} K")
    ok = max(dev.values()) < TOLERANCE_K
    print("PASS" if ok else f"not yet: deviation >= {TOLERANCE_K} K")
    t = result.index.to_numpy(dtype=float)
    fig, ax = plt.subplots()
    ax.plot(hours(t), degc(result["zone.T"]), label="$T_z$ Modelica")
    ax.plot(hours(t), degc(result["envelope.T"]), label="$T_e$ Modelica")
    ax.plot(hours(t), degc(ref.T_z), "k:", label="Python M02")
    ax.plot(hours(t), degc(ref.T_e), "k:")
    ax.set(xlabel="time [h]", ylabel="T [°C]")
    ax.legend()
    return dev

## 2. The starting file, before changing anything

The starting file simulates, but the deviation is about 16 K. Why does an unconnected model
simulate at all? (Hint: what is the value of a flow variable at a port with no connection?)

In [ ]:
dev_start = check(repo_root() / "exercises/session_03/S03_G1_ZoneRC_start.mo")

*Your answer:*

## 3–4. Your model

In OMEdit, declare and connect the missing components (instructions in the card), save the file
under a new name in this folder, and write its name below. Rerun the cell until it prints `PASS`.

In [ ]:
MY_FILE = "my_S03_G1_ZoneRC.mo"  # TODO: your file, in the same folder as this notebook
dev = check(MY_FILE)

## 5. The flat model

Print the flat model and count unknowns and equations. Which equations did each `connect`
create?

In [ ]:
flat = flatten("S03_G1_ZoneRC", [Path(MY_FILE).resolve()], out_dir=OUT, load_library=False)
print(flat)

*Your answer:*